# Cortex DTI — Fresh Kaggle Training Campaign

**Objective.** Conduct a scientifically controlled, genuinely fresh Cortex training campaign on Kaggle T4. This notebook implements Phases A–E as specified.

Preconditions:
- Files copied to Kaggle (repo root contains `src/`, `configs/`, `data/` or splits generated).
- Persistent storage: write artifacts to `/kaggle/working/` and back up to your Dataset (or download) between sessions. `/kaggle/working` is **erased on session restart**.
- T4 (sm_75): use `amp_dtype=fp16`, `drug_tower_fp32=true` (defaults).

## Phase A — Preflight (environment, data, fresh initialisation)

Generate reproducible splits (if not already present in a persistent Dataset), then run preflight against the chosen protocol.

In [ ]:
!cd /kaggle/working/cortex && pip install -q --upgrade pip; pip install -q -r requirements.txt 2>/dev/null || true
# Build/refresh splits (hash-based, deterministic)
!cd /kaggle/working/cortex && python -m src.preprocess --out-dir data/splits/v1 --nonpositive drop
# Preflight
!cd /kaggle/working/cortex && python -m src.preflight \
  --set train_path=data/splits/v1/cold_target/train.parquet \
  --set val_path=data/splits/v1/cold_target/val.parquet \
  --set test_path=data/splits/v1/cold_target/test.parquet \
  --set data_protocol=cold_target --set split_version=v1 \
  --out preflight_report.json

## Phase B — Smoke test (finite loss, checkpoint save/load, interruption safety)

Run a minimal fresh experiment to verify AMP, backward, checkpointing and resume semantics.

In [ ]:
!cd /kaggle/working/cortex && python -m src.train --fresh-run --experiment-name smoke \
  --output-dir experiments \
  --epochs 1 --batch-size 32 --num-workers 2 \
  --limit-train 64 --limit-val 32 --max-steps 4 \
  --set train_path=data/splits/v1/cold_target/train.parquet \
  --set val_path=data/splits/v1/cold_target/val.parquet \
  --set test_path=data/splits/v1/cold_target/test.parquet \
  --set data_protocol=cold_target --set split_version=v1

In [ ]:
# Resume the smoke run (interruption/recovery test)
!cd /kaggle/working/cortex && python -m src.train --resume auto --experiment-name smoke \
  --output-dir experiments \
  --epochs 2 --batch-size 32 --num-workers 2 \
  --limit-train 64 --limit-val 32 --max-steps 6 \
  --set train_path=data/splits/v1/cold_target/train.parquet \
  --set val_path=data/splits/v1/cold_target/val.parquet \
  --set test_path=data/splits/v1/cold_target/test.parquet

## Phase C — Baseline training (genuinely fresh)

Train the fresh baseline on the cleaned cold_target splits. Model selection uses `select_metric=rmse` by default.

In [ ]:
!cd /kaggle/working/cortex && python -m src.train --fresh-run --experiment-name baseline \
  --output-dir experiments \
  --set train_path=data/splits/v1/cold_target/train.parquet \
  --set val_path=data/splits/v1/cold_target/val.parquet \
  --set test_path=data/splits/v1/cold_target/test.parquet \
  --set data_protocol=cold_target --set split_version=v1

## Phase D — Staged optimization campaign

Run the compute-conscious campaign (one factor at a time). Limit to a subset with `--stage` or `--only` if time/quotas are tight.

In [ ]:
# Dry-run to inspect commands
!cd /kaggle/working/cortex && python -m src.campaign \
  --split-dir data/splits/v1 --protocol cold_target --dry-run
# Run stage 1 only (quick)
!cd /kaggle/working/cortex && python -m src.campaign \
  --split-dir data/splits/v1 --protocol cold_target --stage stage1_lr
# To run all stages, comment the stage1_lr line and run without --stage

## Phase E — Final evaluation on the untouched test set

Select the best checkpoint by validation metric (RMSE by default), then evaluate *only* the test split. **Never** use test during model selection.

Replace `<BEST_EXP>` with the chosen experiment directory (e.g. `experiments/baseline/checkpoints/best.pt` or `experiments/campaign_v1/lr_1e-4/checkpoints/best.pt`).

In [ ]:
# Evaluate best checkpoint on test
!cd /kaggle/working/cortex && python -m src.evaluate \
  --checkpoint experiments/baseline/checkpoints/best.pt \
  --split test \
  --json-out reports/baseline_test.json

## Export recovery artifacts (required between sessions)

Bundle checkpoints/configs/metrics to persistent storage (Dataset or download). Exclude large `step_*.pt` to reduce size.

In [ ]:
!mkdir -p /kaggle/working/backups
!cd /kaggle/working/cortex && python -m src.export_run \
  --source experiments --dest /kaggle/working/backups --no-extras